In [2]:
import os

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import get_current_run_tree


In [12]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [4]:
# ── Load + split the real guide document ─────────────────────────────────
loader   = TextLoader("../data/llm_production_guide.txt", encoding="utf-8")
raw_docs = loader.load()
print(f"Loaded: {len(raw_docs[0].page_content):,} characters")

Loaded: 11,679 characters


In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
chunks   = splitter.split_documents(raw_docs)
print(f"Chunks: {len(chunks)}  (avg {sum(len(c.page_content) for c in chunks)//len(chunks)} chars each)")

Chunks: 27  (avg 430 chars each)


In [6]:
print("\nEmbedding chunks via Gemini API (gemini-embedding-2-preview)…")
embeddings  = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅  FAISS index ready")


Embedding chunks via Gemini API (gemini-embedding-2-preview)…
✅  FAISS index ready


## CustomTrace

In [7]:
import re
from langsmith import traceable

from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

# Plain llm.invoke() — no LCEL, no chain, no decorators.
# LangSmith intercepts this call automatically via the three env vars we set.
response = llm.invoke("What is a LangSmith Run? Answer in 2 sentences.")
print(response.content)


A LangSmith Run is a recorded execution of a LangChain (or LangSmith‑enabled) application that logs the inputs, outputs, metadata, and performance metrics of each step in the LLM workflow. It provides developers with a searchable history for debugging, monitoring, and analyzing the behavior of their language‑model pipelines.


In [8]:
# ── Traced RAG function — @traceable, no LCEL ────────────────────────────
@traceable(run_type="chain", name="production_guide_rag")
def rag(question: str, user_id: str = "anonymous") -> str:
    docs    = retriever.invoke(question)
    context = "\n\n".join(f"[chunk {i+1}] {d.page_content}" for i, d in enumerate(docs))
    prompt  = (
        f"Answer based ONLY on the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\nAnswer concisely:"
    )
    run = get_current_run_tree()
    if run:
        run.metadata.update({"user_id": user_id, "chunks_retrieved": len(docs)})
    return llm.invoke(prompt).content

In [9]:
# ── Test with two questions ───────────────────────────────────────────────
for q, uid in [
    ("What are the main LLM security risks in production?", "student_01"),
    ("How should we evaluate LLM outputs for quality?",     "student_02"),
]:
    answer = rag(q, user_id=uid)
    print(f"\nQ: {q}")
    print(f"A: {answer[:250]}...")



Q: What are the main LLM security risks in production?
A: The primary production‑level LLM security risks highlighted in the provided material are:

1. **LLM06 – Sensitive Information Disclosure** – the model leaks private data from its training set, system prompt, or retrieved documents (especially in RAG ...

Q: How should we evaluate LLM outputs for quality?
A: Evaluate LLM outputs systematically by :

1. **Create ground‑truth references** – use domain experts or a stronger model (or program‑verifiable closed‑book questions) to generate correct answers.  
2. **Apply LLM‑as‑Judge** – give the question, the r...


## Agentic Rag

In [10]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage
from langchain_community.utilities import GoogleSerperAPIWrapper

In [13]:
serper = GoogleSerperAPIWrapper()

In [14]:
@tool
def search_local_docs(query: str) -> str:
    """
    Search the internal LLM production guide.

    IMPORTANT:
    - Use only once per question.
    - After receiving results, answer the user.
    - Do not call repeatedly.
    """

    docs = vectorstore.similarity_search(query, k=3)

    if not docs:
        return "No relevant documents found."

    response = "\n\n".join(
        f"[Chunk {i+1}]\n{doc.page_content[:700]}"
        for i, doc in enumerate(docs)
    )

    # Prevent huge context windows
    return response[:2500]

In [15]:
@tool
def google_search(query: str) -> str:
    """
    Search the web for recent information.

    IMPORTANT:
    - Use only once per question.
    - After receiving results, answer the user.
    - Do not search again unless absolutely required.
    """

    try:
        result = serper.run(query)

        if not result:
            return "No search results found."

        return str(result)[:2500]

    except Exception as e:
        return f"Search failed: {str(e)}"


In [16]:
agent = create_agent(
    model=llm,
    tools = [search_local_docs , google_search],
    system_prompt=""" 
    
    You are a research assistant.

    You have two tools:

    1. search_local_docs
    - Use for RAG, security, evaluation, monitoring,
    prompt engineering, guardrails, deployment.

    2. google_search
    - Use for current events, news,
    regulations, recent AI developments.

    Rules:

    1. Call a tool ONLY if needed.
    2. Never call the same tool more than once.
    3. Maximum TWO total tool calls.
    4. After receiving tool results, provide the final answer.
    5. Do NOT continue searching if enough information exists.
    6. Do NOT loop.
    7. If one tool gives sufficient information,
    answer immediately.
    
    """
)

In [17]:
def run_agent(question: str):

    result = agent.invoke(
        {
            "messages": [
                HumanMessage(content=question)
            ]
        },
        config={
            "recursion_limit": 10
        }
    )

    tools_used = []

    for msg in result["messages"]:
        if isinstance(msg, ToolMessage):
            tools_used.append(msg.name)

    final_answer = ""

    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage):
            final_answer = msg.content
            break

    return final_answer, list(dict.fromkeys(tools_used))


In [18]:
queries = [
    (
        "What are LLM prompt injection attacks and how do we defend against them?",
        "search_local_docs"
    ),
    (
        "What are the latest AI regulations passed in 2025?",
        "google_search"
    ),
    (
        "How does RAG work and what are the latest open-source RAG frameworks in 2025?",
        "both"
    )
]

In [19]:
for question, expected in queries:

    print("\n" + "=" * 80)
    print("QUESTION:")
    print(question)

    print("\nEXPECTED:")
    print(expected)

    answer, tools = run_agent(question)

    print("\nTOOLS USED:")
    print(tools)

    print("\nANSWER:")
    print(answer[:500])

print("\n✅ Completed successfully")


QUESTION:
What are LLM prompt injection attacks and how do we defend against them?

EXPECTED:
search_local_docs

TOOLS USED:
['search_local_docs']

ANSWER:
**What a “prompt‑injection” attack is**

A prompt‑injection attack is a class of adversarial inputs that tries to hijack or subvert the behavior of a large language model (LLM) by inserting malicious instructions into the user‑supplied prompt (or into any text the model later consumes).  

| Type | How it works | Typical effect |
|------|--------------|----------------|
| **Direct injection** | The attacker appends a command to the user message that overrides the system prompt, e.g. “*Ignore all

QUESTION:
What are the latest AI regulations passed in 2025?

EXPECTED:
google_search

TOOLS USED:
['google_search']

ANSWER:
**Key AI‑related regulations that were enacted or took effect in 2025**

| Date (2025) | Jurisdiction / Authority | Regulation / Executive Order | Core Requirements & Scope |
|-------------|-------------------------